# Visibility into TrustCall JSON Patch

**Trustcall creates and updates JSON schemas. We may need to access and observ the updates made . TrustCall provides Json doc Id everytime the JSON patch is created. 
That is, when TrustCall updates an existing memory.**

### visibility into the specific changes made by Trustcall?
- we saw before that Trustcall has some of its own tools to:
    - Self-correct from validation failures.
    - Update existing documents

Visibility into these tools can be useful for the agent we're going to build.

## Goal: 

In [1]:
# Load API key
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GOOGLE_API_USE_V1"] = "true"


In [2]:
# create genai client and llm
from google import genai

client = genai.Client(api_key = os.environ["GOOGLE_API_KEY"])
for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr

In [26]:
# create a llm using any of the above models
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI( model= "gemini-2.5-flash" , 
                              temperature = 0.2 )
llm.invoke("What day is this?").content

'Today is **Wednesday, May 15, 2024**.'

## create Memory Schema


In [7]:
from pydantic import BaseModel , Field
from typing import List

class Memory(BaseModel):
    content: str = Field(description="Collect information about the user. Example: User name is Dan. User likes to plan a trip to Florida")

class memory_Collection(BaseModel):
    memory: List[Memory] =Field(description= "List of memory contents about the user")



### create a spy class to tools observability

In [8]:
# Inspect the tool calls made by Trustcall
class Spy:
    def __init__(self):
        self.called_tools = []

    def __call__(self, run):
        # Collect information about the tool calls made by the extractor.
        q = [run]
        while q:
            r = q.pop()
            if r.child_runs:
                q.extend(r.child_runs)
            if r.run_type == "chat_model":
                self.called_tools.append(
                    r.outputs["generations"][0][0]["message"]["kwargs"]["tool_calls"]
                )

### Add spy to the TrustCall `with_listeners` extension
```
# Create the extractor
trustcall_extractor = create_extractor(
    model,
    tools=[Memory],
    tool_choice="Memory",
    enable_inserts=True,
)
# Add the spy as a listener
trustcall_extractor_see_all_tool_calls = trustcall_extractor.with_listeners(on_end=spy)
```
## OR 


### Create the extractor
```
trustcall_extractor = create_extractor(
    model,
    tools=[Memory],
    tool_choice="Memory",
    enable_inserts=True,
    Listners = [spy]
)
```

In [27]:
# trustcall with listeners
from trustcall import create_extractor

# Create the extractor
trustcall_extractor = create_extractor(
    llm,
    tools=[Memory],
    tool_choice="Memory",
    enable_inserts=True,
)
spy = Spy()
# Add the spy as a listener
trustcall_extractor_see_all_tool_calls = trustcall_extractor.with_listeners(on_end=spy)

### Experiment with trustcall

In [13]:
from langchain_core.messages import AIMessage, SystemMessage , HumanMessage

conversation = [HumanMessage(content="Hi, I'm Diya. I wanted to know if there are any local Libraries here in Irving"), 
                AIMessage(content="Nice to meet you.  There should be. Which state do you live in? "),
                HumanMessage(content=" I live in Dallas, TX.  I want to go eat first")]

sys_inst = SystemMessage(content= "You are a memory extracor. extract user information from the following conversation.")
# extract memory from the conversation
memory_extracted = trustcall_extractor.invoke({'messages': [sys_inst] + conversation})


In [14]:
memory_extracted

{'messages': [AIMessage(content=[], additional_kwargs={'function_call': {'name': 'Memory', 'arguments': '{"content": "User\'s name is Diya. User lives in Dallas, TX. User is interested in libraries in Irving, TX. User wants to eat before visiting a library."}'}, '__gemini_function_call_thought_signatures__': {'2149f44d-db2a-482b-aa55-86366d8b172f': 'EjQKMgEMOdbHLs9acgdpnV1N2YPAzBK0+jGYULwIKaJ+wNeFDB8/KYKHm8roxISRXQ06HjCd'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019eae2e-9b9b-76c2-b9e0-46ed8e6409e0-0', tool_calls=[{'name': 'Memory', 'args': {'content': "User's name is Diya. User lives in Dallas, TX. User is interested in libraries in Irving, TX. User wants to eat before visiting a library."}, 'id': '2149f44d-db2a-482b-aa55-86366d8b172f', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 133, 'output_tokens': 47, 'total_tokens': 180, 'input_token_

In [16]:
for m in memory_extracted['messages']:
    m.pretty_print()

================================== Ai Message ==================================

[]
Tool Calls:
  Memory (2149f44d-db2a-482b-aa55-86366d8b172f)
 Call ID: 2149f44d-db2a-482b-aa55-86366d8b172f
  Args:
    content: User's name is Diya. User lives in Dallas, TX. User is interested in libraries in Irving, TX. User wants to eat before visiting a library.


In [20]:
memory_extracted['responses'][-1].model_dump()

{'content': "User's name is Diya. User lives in Dallas, TX. User is interested in libraries in Irving, TX. User wants to eat before visiting a library."}

## continue conversation and create new or update existing memory with listner

In [21]:
# get existing memory from perevious trustcall 'responses'
# existing memory is a list of tuple : (key , tool_name , value)
existing_memory = [(str(i) , 'Memory' , m.model_dump()) for i, m in  enumerate(memory_extracted['responses'])]if memory_extracted['responses'] else None
existing_memory

[('0',
  'Memory',
  {'content': "User's name is Diya. User lives in Dallas, TX. User is interested in libraries in Irving, TX. User wants to eat before visiting a library."})]

In [36]:
#  continue converation and extract memory

updated_conversation = [AIMessage(content="Thats great! You can visit jona's Library in Irving. There is a bekery near by called 'Corner shop' where you can try breakfast and snacks.")
                        ,
                        HumanMessage(content="Nice. I'll try 'Corner Shop' bekery and have some coffee with waffles. I will go bike riding later"),
                       ]

sys_info = SystemMessage(content= "Update existing memories and also create new memory based on the given conversation below. perform both action")

# get listner
New_memory = trustcall_extractor_see_all_tool_calls.invoke({'messages': [sys_info] + updated_conversation , 
                                         'existing': existing_memory })
New_memory

{'messages': [AIMessage(content='', additional_kwargs={'updated_docs': {'64b7aa23-3e70-47b5-8f8c-ffba0aef99f9': '0'}}, response_metadata={}, id='d7e563b7-7f6a-4597-949c-93ad0cdb0a22', tool_calls=[{'name': 'Memory', 'args': {'content': "User's name is Diya. User lives in Dallas, TX. User is interested in libraries in Irving, TX. User wants to eat coffee with waffles at 'Corner Shop' bakery before visiting a library."}, 'id': '64b7aa23-3e70-47b5-8f8c-ffba0aef99f9', 'type': 'tool_call'}, {'name': 'Memory', 'args': {'content': 'User will go bike riding later.'}, 'id': '76984170-0c52-4ead-9b13-9cb13ef57682', 'type': 'tool_call'}], invalid_tool_calls=[])],
 'responses': [Memory(content="User's name is Diya. User lives in Dallas, TX. User is interested in libraries in Irving, TX. User wants to eat coffee with waffles at 'Corner Shop' bakery before visiting a library."),
  Memory(content='User will go bike riding later.')],
 'response_metadata': [{'id': '64b7aa23-3e70-47b5-8f8c-ffba0aef99f9',


In [37]:
# Messages contain the tool calls
for m in New_memory["messages"]:
    m.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  Memory (64b7aa23-3e70-47b5-8f8c-ffba0aef99f9)
 Call ID: 64b7aa23-3e70-47b5-8f8c-ffba0aef99f9
  Args:
    content: User's name is Diya. User lives in Dallas, TX. User is interested in libraries in Irving, TX. User wants to eat coffee with waffles at 'Corner Shop' bakery before visiting a library.
  Memory (76984170-0c52-4ead-9b13-9cb13ef57682)
 Call ID: 76984170-0c52-4ead-9b13-9cb13ef57682
  Args:
    content: User will go bike riding later.


In [41]:
for m in New_memory['responses']:
    print(m.model_dump())

{'content': "User's name is Diya. User lives in Dallas, TX. User is interested in libraries in Irving, TX. User wants to eat coffee with waffles at 'Corner Shop' bakery before visiting a library."}
{'content': 'User will go bike riding later.'}


In [42]:
# Inspect the tool calls made by Trustcall
spy.called_tools

[[{'name': 'PatchDoc',
   'args': {'planned_edits': "The user wants to try 'Corner Shop' bakery, have coffee with waffles, and go bike riding later. I will update the 'content' field of the Memory instance to include this new information.",
    'patches': [{'path': '/content',
      'value': "User's name is Diya. User lives in Dallas, TX. User is interested in libraries in Irving, TX. User wants to eat before visiting a library. User will try 'Corner Shop' bakery and have coffee with waffles. User will go bike riding later.",
      'op': 'replace'}],
    'json_doc_id': '0'},
   'id': 'f2ba7572-99c1-4ff0-b257-e3d0394bdf12',
   'type': 'tool_call'}],
 [{'name': 'PatchDoc',
   'args': {'patches': [{'value': "User's name is Diya. User lives in Dallas, TX. User is interested in libraries in Irving, TX. User wants to eat before visiting a library. User will try 'Corner Shop' bakery and have coffee with waffles. User will go bike riding later.",
      'path': '/content',
      'op': 'replace'

## extract information form the spy tool calls for documentation

In [68]:
# create a function to extract information from spy.tool_calls

def extract_tool_info(tool_calls_list , schema_name):
    ''' Extract tool information from apy.tool_Calls for observability
        Args:
            tool_calls: List of tool calls from the model
            schema_name: Name of the schema tool (e.g., "Memory", "ToDo", "Profile") '''
    docs= []
    for tool_call in tool_calls_list:
        for call in tool_call:
            if call['name'] == 'PatchDoc':
                doc = f"""Json patch applied:
                - Memory_document id: {call['args']['json_doc_id']}.. updated
                - Planned_content: {call['args']['planned_edits']} .. created
                - Acutal_content: {call['args']['patches'][0].get('value',None)}..applied\n\n"""
                docs.append(doc+"-"*30)
            elif call['name'] == schema_name:
                doc = f"""New {schema_name} created:
                      - New_memory: {call['args']}\n\n"""
                docs.append(doc+"-"*30)

    return docs

Schema_name = "Memory"
spy_info = spy.called_tools
Changes_Observed = extract_tool_info(spy_info , Schema_name)
Changes_Observed                                  

["Json patch applied:\n                - Memory_document id: 0.. updated\n                - Planned_content: The user wants to try 'Corner Shop' bakery, have coffee with waffles, and go bike riding later. I will update the 'content' field of the Memory instance to include this new information. .. created\n                - Acutal_content: User's name is Diya. User lives in Dallas, TX. User is interested in libraries in Irving, TX. User wants to eat before visiting a library. User will try 'Corner Shop' bakery and have coffee with waffles. User will go bike riding later...applied\n\n------------------------------",
 "Json patch applied:\n                - Memory_document id: 0.. updated\n                - Planned_content: The user is providing new information about their plans. I need to update the existing memory to reflect these new details. I will update the 'content' field to include the new information about trying 'Corner Shop' bakery, having coffee with waffles, and going bike ri

In [69]:
for doc in Changes_Observed:
    print(doc)

Json patch applied:
                - Memory_document id: 0.. updated
                - Planned_content: The user wants to try 'Corner Shop' bakery, have coffee with waffles, and go bike riding later. I will update the 'content' field of the Memory instance to include this new information. .. created
                - Acutal_content: User's name is Diya. User lives in Dallas, TX. User is interested in libraries in Irving, TX. User wants to eat before visiting a library. User will try 'Corner Shop' bakery and have coffee with waffles. User will go bike riding later...applied

------------------------------
Json patch applied:
                - Memory_document id: 0.. updated
                - Planned_content: The user is providing new information about their plans. I need to update the existing memory to reflect these new details. I will update the 'content' field to include the new information about trying 'Corner Shop' bakery, having coffee with waffles, and going bike riding later. I